# Investigating Non-Local Coding via Partially Overlapping Flight Trajectories

Find pairs of flight trajectory clusters that share a contiguous spatial segment but diverge elsewhere. By comparing neural activity in the shared vs. divergent portions, we test whether hippocampal neurons encode more than just current position (non-local / trajectory-dependent coding).

**Approach:**
- Load raw .mat files directly (no z-scoring, positions in meters)
- Use existing `FlightClusters` to get cluster labels
- Sliding-window Frechet distance on cluster-mean trajectories to identify shared segments
- Align raw spike times to shared segment onset
- Permutation tests: position should NOT differ in shared segment; PSTHs may differ

In [ ]:
import sys
sys.path.insert(0, '/home/alowet/Documents/code/digitalbat')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from scipy.io import loadmat
from scipy.interpolate import interp1d
from scipy.ndimage import label as ndlabel
from itertools import combinations
import glob

from digitalbat.utility.utils import frechet_distance, compute_fractional_abscissa, FlightClusters

sns.set_theme(style='whitegrid', context='notebook')

# --- Constants ---
PARENT_FOLDER = '/mnt/s5/Adam/Angelo/NP_datasets/'
SESSION_NAMES = [
    'Dataset_3_14543_240419',
    'Dataset_3_14543_240420',
    'Dataset_3_14543_240421',
    'Dataset_3_29959_240402',
    'Dataset_3_29959_240403',
    'Dataset_3_29959_240404',
    'Dataset_2_14445_231208',
    'Dataset_2_14445_231209',
    'Dataset_1_32622_231009',
    'Dataset_1_32622_231010',
]
MIN_FLIGHT_DUR = 1.0   # seconds — minimum flight duration to keep
N_DS = 100              # downsample resolution for trajectory comparison
WINDOW_FRAC = 0.30      # sliding window = 30% of trajectory
MIN_CLUSTER_SIZE = 3    # minimum flights per cluster for comparison

## Helper Functions

In [ ]:
def load_session_data(session_name, parent_folder=PARENT_FOLDER):
    """Load raw behavior + spike data from .mat files for one session.
    
    Uses Data_Prep_for_Raw for behavior loading and spike merging logic,
    but returns raw spike times (not binned) for raster plots.
    """
    from digitalbat.datasets.data import Data_Prep_for_Raw

    session_folder = parent_folder + session_name
    su_file_path = parent_folder + 'To_transfer/out_Gunits_probe*_' + session_name + '_with_metrics.mat'

    # Use Data_Prep_for_Raw just to get behavior and spike infrastructure
    prep = Data_Prep_for_Raw(
        parent_folder=session_folder,
        su_file_path=su_file_path,
        use_speed_head_dir=False,
        use_IMU_flag=False,
        use_click_flag=False,
        use_LFP_flag=False,
        use_ripple_flag=False,
        only_flight_flag=False,  # we want all data, we'll segment flights ourselves
        spike_count_flag=True,
        bin_width=0.01,  # irrelevant for us but required
    )
    # Load behavior (sets up bhv dict internally)
    prep.get_all_behavior()

    # Load raw behavior .mat for direct access to positions/velocity at native Fs
    bhv_file = glob.glob(session_folder + '/Extracted_Behavior_*.mat')[0]
    bhv = loadmat(bhv_file)
    r = bhv['r']                        # (N, 3) positions in meters
    if 'Dataset_2' in session_name:
        t_bhv = bhv['t'][:, 0]
    else:
        t_bhv = bhv['t'][0, :]
    bflying = bhv['bflying'][:, 0].astype(int)
    Fs = float(bhv['Fs'][0, 0])
    v = bhv['v']                        # (N, 3) velocity in m/s
    r_lim = bhv['r_lim']

    # --- Raw spike times (reuse Data_Prep_for_Raw's loading + merge logic) ---
    # The spike loading/merging code in get_all_spike() bins spikes,
    # but we need raw times. Replicate just the load+merge part.
    su_paths = sorted(glob.glob(su_file_path))
    su_goods_time = []
    su_goods_cluster_id = []
    su_goods_pType = []
    for p in su_paths:
        su = loadmat(p)
        gu = su['out_Gunits']
        su_goods_cluster_id.append(
            np.array([x[0][0, 0] for x in gu['cluster_id']]))
        su_goods_time.append(
            [x[0][:, 0] / 1e6 for x in gu['spikeTimes_usec']])
        su_goods_pType.append(list(np.array(
            [np.squeeze(x[0]).item() if np.squeeze(x[0]).ndim == 0
             else np.squeeze(x[0]) for x in gu['pType']], dtype=object)))

    # Merge units (same logic as Data_Prep_for_Raw.get_all_spike)
    merges_file = glob.glob(session_folder + '/merges_Dataset_*.mat')[0]
    merges = loadmat(merges_file)
    refractory = 0.001
    for ii in range(len(su_paths)):
        if len(merges['merges'][ii][0]):
            cids_to_merge = merges['merges'][ii][0][0][0][0]
            cid_list = su_goods_cluster_id[ii]
            time_list = su_goods_time[ii]
            merge_idx = [np.where(cid_list == cid)[0][0] for cid in cids_to_merge]
            merged = np.sort(np.concatenate([time_list[idx] for idx in merge_idx]))
            clean = [merged[0]]
            for spk_t in merged[1:]:
                if spk_t - clean[-1] > refractory:
                    clean.append(spk_t)
            time_list[merge_idx[0]] = np.array(clean)
            for idx in merge_idx[1:]:
                time_list[idx] = np.array([])
            valid = [i for i, st in enumerate(time_list) if len(st) > 0]
            su_goods_time[ii] = [time_list[i] for i in valid]
            su_goods_cluster_id[ii] = cid_list[valid]
            su_goods_pType[ii] = [su_goods_pType[ii][i] for i in valid]

    spike_times = [st for probe in su_goods_time for st in probe]
    pType = np.array([pt for probe in su_goods_pType for pt in probe], dtype=object)

    return dict(r=r, t=t_bhv, bflying=bflying, Fs=Fs, v=v, r_lim=r_lim,
                spike_times=spike_times, pType=pType)


def segment_flights(bflying, Fs, min_dur=MIN_FLIGHT_DUR):
    """Find flight start/stop indices from binary bflying vector.
    Returns arrays of start and stop indices (inclusive)."""
    bfly = bflying.astype(int)
    starts = np.where(np.diff(bfly) > 0)[0] + 1
    stops = np.where(np.diff(bfly) < 0)[0]
    if bfly[0] > 0:
        starts = np.concatenate([[0], starts])
    if bfly[-1] > 0:
        stops = np.concatenate([stops, [len(bfly) - 1]])
    n = min(len(starts), len(stops))
    starts, stops = starts[:n], stops[:n]
    durs = (stops - starts) / Fs
    keep = durs >= min_dur
    return starts[keep], stops[keep]


def downsample_trajectory(r, f_start, f_stop, n_ds=N_DS):
    """Downsample a single flight trajectory to n_ds evenly spaced points."""
    f_start, f_stop = int(f_start), int(f_stop)
    traj = r[f_start:f_stop + 1]
    t_orig = np.linspace(0, 1, traj.shape[0])
    t_ds = np.linspace(0, 1, n_ds)
    return np.column_stack([
        interp1d(t_orig, traj[:, d], kind='linear')(t_ds) for d in range(3)])


def downsample_velocity(v, f_start, f_stop, n_ds=N_DS):
    """Downsample a single flight velocity to n_ds evenly spaced points."""
    f_start, f_stop = int(f_start), int(f_stop)
    vel = v[f_start:f_stop + 1]
    t_orig = np.linspace(0, 1, vel.shape[0])
    t_ds = np.linspace(0, 1, n_ds)
    return np.column_stack([
        interp1d(t_orig, vel[:, d], kind='linear')(t_ds) for d in range(3)])


def sliding_window_frechet(traj_a, traj_b, window_frac=WINDOW_FRAC):
    """Compute sliding-window Frechet distance between two (N_DS, 3) trajectories."""
    n = len(traj_a)
    half_win = int(n * window_frac) // 2
    local_frechet = np.full(n, np.nan)
    for i in range(n):
        lo = max(0, i - half_win)
        hi = min(n, i + half_win + 1)
        local_frechet[i] = frechet_distance(traj_a[lo:hi], traj_b[lo:hi])
    return local_frechet


def find_shared_segment(local_frechet, threshold):
    """Find the largest contiguous segment where local_frechet < threshold.
    Returns (start_idx, end_idx) or None."""
    below = (local_frechet < threshold).astype(int)
    labeled, n_components = ndlabel(below)
    if n_components == 0:
        return None
    best_label, best_size = 0, 0
    for lbl in range(1, n_components + 1):
        size = np.sum(labeled == lbl)
        if size > best_size:
            best_label, best_size = lbl, size
    indices = np.where(labeled == best_label)[0]
    return indices[0], indices[-1]


print("Helper functions defined.")


## Load Data & Cluster Flights (single session)

In [ ]:
# Pick one session to start
session_name = SESSION_NAMES[0]  # Dataset_3_14543_240419
print(f"Loading {session_name}...")
data = load_session_data(session_name)

r, t, Fs, v = data['r'], data['t'], data['Fs'], data['v']
spike_times, pType = data['spike_times'], data['pType']
print(f"  Positions: {r.shape}, Fs={Fs} Hz, {len(spike_times)} units")

# Cluster flights using existing FlightClusters
fc = FlightClusters(r_vector=r, f_vector=data['bflying'], Fs=Fs,
                    frechet_clus=True, alpha=0.8)
fc_results = fc.compute()
f_starts, f_stops = fc_results['start_frame'], fc_results['stop_frame']
cluster_ids = fc_results['id']
n_flights = fc_results['N']

# Build df_flights
records = []
for i in range(n_flights):
    dur = (f_stops[i] - f_starts[i]) / Fs
    if dur < MIN_FLIGHT_DUR:
        continue
    records.append(dict(
        flight_idx=i,
        cluster_id=cluster_ids[i],
        f_start=f_starts[i],
        f_stop=f_stops[i],
        duration=dur,
        n_samples=f_stops[i] - f_starts[i] + 1,
    ))
df_flights = pd.DataFrame(records)
print(f"\n{len(df_flights)} flights after filtering (>= {MIN_FLIGHT_DUR}s)")
print(df_flights.groupby('cluster_id').size().rename('n_flights').reset_index())

# Compute mean trajectory and velocity per cluster (downsampled to N_DS)
cluster_records = []
for cid, grp in df_flights.groupby('cluster_id'):
    if len(grp) < MIN_CLUSTER_SIZE:
        continue
    trajs_ds = np.stack([downsample_trajectory(r, row.f_start, row.f_stop)
                         for _, row in grp.iterrows()])  # (n_flights, N_DS, 3)
    vels_ds = np.stack([downsample_velocity(v, row.f_start, row.f_stop)
                        for _, row in grp.iterrows()])   # (n_flights, N_DS, 3)
    cluster_records.append(dict(
        cluster_id=cid,
        n_flights=len(grp),
        mean_traj=np.nanmean(trajs_ds, axis=0),    # (N_DS, 3)
        mean_vel=np.nanmean(vels_ds, axis=0),       # (N_DS, 3)
    ))

df_clusters = pd.DataFrame(cluster_records)
print(f"\n{len(df_clusters)} clusters with >= {MIN_CLUSTER_SIZE} flights:")
print(df_clusters[['cluster_id', 'n_flights']])

## Find Partially Overlapping Cluster Pairs

In [ ]:
# Compute sliding-window Frechet between all cluster pairs
pair_records = []
cluster_ids_list = df_clusters['cluster_id'].values
mean_trajs = df_clusters['mean_traj'].values
mean_vels = df_clusters['mean_vel'].values

for i, j in combinations(range(len(cluster_ids_list)), 2):
    lf_pos = sliding_window_frechet(mean_trajs[i], mean_trajs[j])
    lf_vel = sliding_window_frechet(mean_vels[i], mean_vels[j])
    pair_records.append(dict(
        cluster_a=cluster_ids_list[i],
        cluster_b=cluster_ids_list[j],
        n_flights_a=df_clusters.iloc[i]['n_flights'],
        n_flights_b=df_clusters.iloc[j]['n_flights'],
        local_frechet_pos=lf_pos,
        local_frechet_vel=lf_vel,
        global_frechet=frechet_distance(mean_trajs[i], mean_trajs[j]),
    ))

df_pairs = pd.DataFrame(pair_records)
print(f"{len(df_pairs)} cluster pairs to evaluate")

In [ ]:
# Threshold sweep: how many qualifying pairs at each distance threshold?
all_lf = np.concatenate(df_pairs['local_frechet_pos'].values)
thresholds = np.linspace(np.nanpercentile(all_lf, 10), np.nanpercentile(all_lf, 90), 50)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: distribution of all local Frechet values
sns.histplot(all_lf[~np.isnan(all_lf)], bins=80, ax=axes[0])
axes[0].set_xlabel('Local Frechet distance (m)')
axes[0].set_title('Distribution of sliding-window Frechet distances')

# Right: number of qualifying pairs vs threshold
n_qualifying = []
for th in thresholds:
    count = 0
    for _, row in df_pairs.iterrows():
        seg = find_shared_segment(row['local_frechet_pos'], th)
        if seg is not None:
            overlap_frac = (seg[1] - seg[0] + 1) / N_DS
            if 0.3 <= overlap_frac <= 0.7:
                count += 1
    n_qualifying.append(count)

axes[1].plot(thresholds, n_qualifying, 'o-', markersize=3)
axes[1].set_xlabel('Distance threshold (m)')
axes[1].set_ylabel('# qualifying pairs (30-70% overlap)')
axes[1].set_title('Threshold sweep')
sns.despine()
plt.tight_layout()
plt.show()

# Print qualifying pairs at a few thresholds
for th in [0.3, 0.5, 0.7, 1.0]:
    hits = []
    for _, row in df_pairs.iterrows():
        seg = find_shared_segment(row['local_frechet_pos'], th)
        if seg is not None:
            overlap_frac = (seg[1] - seg[0] + 1) / N_DS
            if 0.3 <= overlap_frac <= 0.7:
                hits.append((row['cluster_a'], row['cluster_b'], overlap_frac, seg))
    print(f"  threshold={th:.1f}m: {len(hits)} pairs — {hits}")

In [ ]:
# --- Select threshold and find qualifying pairs ---
DIST_THRESHOLD = 0.5  # <-- adjust based on sweep above

qualifying = []
for idx, row in df_pairs.iterrows():
    seg = find_shared_segment(row['local_frechet_pos'], DIST_THRESHOLD)
    if seg is None:
        continue
    overlap_frac = (seg[1] - seg[0] + 1) / N_DS
    if 0.3 <= overlap_frac <= 0.7:
        qualifying.append(dict(
            pair_idx=idx,
            cluster_a=row['cluster_a'],
            cluster_b=row['cluster_b'],
            n_flights_a=row['n_flights_a'],
            n_flights_b=row['n_flights_b'],
            shared_start=seg[0],
            shared_end=seg[1],
            overlap_frac=overlap_frac,
            shared_start_frac=seg[0] / N_DS,
            shared_end_frac=seg[1] / N_DS,
        ))

df_qualifying = pd.DataFrame(qualifying)
print(f"{len(df_qualifying)} qualifying pairs at threshold={DIST_THRESHOLD}m")
df_qualifying

## Visualize Candidate Pairs

In [ ]:
pal = sns.color_palette('tab10')

for _, qrow in df_qualifying.iterrows():
    pair_row = df_pairs.iloc[qrow['pair_idx']]
    ca, cb = int(qrow['cluster_a']), int(qrow['cluster_b'])
    s_start, s_end = int(qrow['shared_start']), int(qrow['shared_end'])

    # Get mean trajectories
    traj_a = df_clusters.loc[df_clusters['cluster_id'] == ca, 'mean_traj'].values[0]
    traj_b = df_clusters.loc[df_clusters['cluster_id'] == cb, 'mean_traj'].values[0]
    lf_pos = pair_row['local_frechet_pos']
    lf_vel = pair_row['local_frechet_vel']

    fig = plt.figure(figsize=(16, 8))

    # --- 3D trajectory plot ---
    ax3d = fig.add_subplot(2, 2, 1, projection='3d')
    frac = np.linspace(0, 1, N_DS)

    # Shared segment mask
    shared = np.zeros(N_DS, dtype=bool)
    shared[s_start:s_end + 1] = True

    for traj, color, label in [(traj_a, pal[0], f'Cluster {ca}'),
                                (traj_b, pal[1], f'Cluster {cb}')]:
        # Divergent portions
        ax3d.plot(traj[~shared, 0], traj[~shared, 1], traj[~shared, 2],
                  '.', color=color, alpha=0.4, markersize=3)
        # Shared portion
        ax3d.plot(traj[shared, 0], traj[shared, 1], traj[shared, 2],
                  '-', color=color, alpha=0.9, linewidth=2, label=label)

    # Highlight shared segment start/end
    for traj in [traj_a, traj_b]:
        ax3d.scatter(*traj[s_start], color='green', s=60, zorder=5, marker='o')
        ax3d.scatter(*traj[s_end], color='red', s=60, zorder=5, marker='s')

    ax3d.set_xlabel('X (m)')
    ax3d.set_ylabel('Y (m)')
    ax3d.set_zlabel('Z (m)')
    ax3d.set_title(f'Clusters {ca} vs {cb} — {qrow["overlap_frac"]:.0%} shared')
    ax3d.legend(fontsize=8)

    # --- 2D top-down view ---
    ax2d = fig.add_subplot(2, 2, 2)
    for traj, color, label in [(traj_a, pal[0], f'Cluster {ca}'),
                                (traj_b, pal[1], f'Cluster {cb}')]:
        ax2d.plot(traj[:, 0], traj[:, 1], '-', color=color, alpha=0.3, linewidth=1)
        ax2d.plot(traj[shared, 0], traj[shared, 1], '-', color=color, alpha=0.9,
                  linewidth=3, label=label + ' (shared)')
    ax2d.set_xlabel('X (m)')
    ax2d.set_ylabel('Y (m)')
    ax2d.set_title('Top-down view (XY)')
    ax2d.legend(fontsize=8)
    ax2d.set_aspect('equal')

    # --- Sliding-window Frechet distance ---
    ax_lf = fig.add_subplot(2, 2, 3)
    ax_lf.plot(frac, lf_pos, color='k', label='Position Frechet')
    ax_lf.plot(frac, lf_vel, color='gray', alpha=0.6, label='Velocity Frechet')
    ax_lf.axhline(DIST_THRESHOLD, color='r', linestyle='--', alpha=0.7, label=f'Threshold={DIST_THRESHOLD}m')
    ax_lf.axvspan(frac[s_start], frac[s_end], color='green', alpha=0.15, label='Shared segment')
    ax_lf.set_xlabel('Fractional trajectory position')
    ax_lf.set_ylabel('Frechet distance (m)')
    ax_lf.set_title('Sliding-window Frechet distance')
    ax_lf.legend(fontsize=7)

    # --- Velocity magnitude comparison ---
    ax_vel = fig.add_subplot(2, 2, 4)
    vel_mag_a = np.linalg.norm(
        df_clusters.loc[df_clusters['cluster_id'] == ca, 'mean_vel'].values[0], axis=1)
    vel_mag_b = np.linalg.norm(
        df_clusters.loc[df_clusters['cluster_id'] == cb, 'mean_vel'].values[0], axis=1)
    ax_vel.plot(frac, vel_mag_a, color=pal[0], label=f'Cluster {ca}')
    ax_vel.plot(frac, vel_mag_b, color=pal[1], label=f'Cluster {cb}')
    ax_vel.axvspan(frac[s_start], frac[s_end], color='green', alpha=0.15)
    ax_vel.set_xlabel('Fractional trajectory position')
    ax_vel.set_ylabel('Speed (m/s)')
    ax_vel.set_title('Velocity magnitude')
    ax_vel.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

## Align Individual Flights to Shared Segment

In [ ]:
# Select the first qualifying pair (or change index to explore others)
qrow = df_qualifying.iloc[0]
ca, cb = int(qrow['cluster_a']), int(qrow['cluster_b'])
shared_start_frac = qrow['shared_start_frac']
shared_end_frac = qrow['shared_end_frac']
print(f"Selected pair: clusters {ca} vs {cb}, shared segment: {shared_start_frac:.2f}–{shared_end_frac:.2f}")

# For each flight in both clusters, find the raw time index of the shared segment
aligned_flights = []
for _, frow in df_flights[df_flights['cluster_id'].isin([ca, cb])].iterrows():
    f_start, f_stop = frow['f_start'], frow['f_stop']
    flight_r = r[f_start:f_stop + 1]  # (T, 3)
    flight_v = v[f_start:f_stop + 1]  # (T, 3)
    flight_t = t[f_start:f_stop + 1]  # (T,)

    # Fractional abscissa for this flight
    frac_abs = compute_fractional_abscissa(flight_r)  # (T,) in [0, 1]

    # Map shared segment boundaries to raw indices
    shared_start_idx = np.searchsorted(frac_abs, shared_start_frac, side='left')
    shared_end_idx = np.searchsorted(frac_abs, shared_end_frac, side='right') - 1
    shared_start_idx = np.clip(shared_start_idx, 0, len(frac_abs) - 1)
    shared_end_idx = np.clip(shared_end_idx, 0, len(frac_abs) - 1)

    # Time of shared segment start (for alignment)
    t_shared_start = flight_t[shared_start_idx]

    # Aligned time: t=0 at shared segment start
    t_aligned = flight_t - t_shared_start

    # Shared mask
    shared_mask = np.zeros(len(flight_t), dtype=bool)
    shared_mask[shared_start_idx:shared_end_idx + 1] = True

    aligned_flights.append(dict(
        cluster_id=frow['cluster_id'],
        flight_idx=frow['flight_idx'],
        f_start=f_start,
        f_stop=f_stop,
        t_aligned=t_aligned,
        t_abs=flight_t,
        r=flight_r,
        v=flight_v,
        shared_mask=shared_mask,
        shared_start_idx=shared_start_idx,
        shared_end_idx=shared_end_idx,
        t_shared_start=t_shared_start,
        t_shared_end=flight_t[shared_end_idx],
        shared_dur=flight_t[shared_end_idx] - t_shared_start,
    ))

df_aligned = pd.DataFrame(aligned_flights)
print(f"\n{len(df_aligned)} aligned flights:")
print(df_aligned.groupby('cluster_id')[['shared_dur']].describe().round(3))

## Raster Plots with Position Traces

In [ ]:
# Align spike times for each flight
for i, row in df_aligned.iterrows():
    t_start_abs = row['t_abs'][0]           # absolute time of flight start
    t_end_abs = row['t_abs'][-1]            # absolute time of flight end
    t_offset = row['t_shared_start']        # absolute time of shared segment start

    spikes_aligned = []
    for unit_spk in spike_times:
        # Get spikes during this flight
        mask = (unit_spk >= t_start_abs) & (unit_spk <= t_end_abs)
        spikes_aligned.append(unit_spk[mask] - t_offset)  # align to shared start
    df_aligned.at[i, 'spikes_aligned'] = spikes_aligned

# Select a few neurons to plot (pick units with highest firing rate during these flights)
n_units = len(spike_times)
unit_rates = np.zeros(n_units)
for _, row in df_aligned.iterrows():
    dur = row['t_abs'][-1] - row['t_abs'][0]
    for u in range(n_units):
        unit_rates[u] += len(row['spikes_aligned'][u]) / dur
unit_rates /= len(df_aligned)

# Top 6 most active neurons
top_units = np.argsort(unit_rates)[::-1][:6]
print(f"Plotting top 6 units by mean firing rate: {top_units}")
print(f"  Firing rates: {unit_rates[top_units].round(1)} Hz")

In [ ]:
def plot_raster_with_positions(df_aligned, unit_idx, ca, cb, pal=pal):
    """Plot raster for one neuron + position traces, aligned to shared segment start."""
    flights_a = df_aligned[df_aligned['cluster_id'] == ca].reset_index(drop=True)
    flights_b = df_aligned[df_aligned['cluster_id'] == cb].reset_index(drop=True)
    n_a, n_b = len(flights_a), len(flights_b)

    fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True,
                              gridspec_kw={'height_ratios': [2, 1, 1, 1]})

    # Compute time range for x-axis
    all_t_min = min(row['t_aligned'][0] for _, row in df_aligned.iterrows())
    all_t_max = max(row['t_aligned'][-1] for _, row in df_aligned.iterrows())

    # --- Raster plot ---
    ax = axes[0]
    trial = 0
    for cluster_flights, color, label in [(flights_a, pal[0], f'Cluster {ca}'),
                                           (flights_b, pal[1], f'Cluster {cb}')]:
        for _, row in cluster_flights.iterrows():
            spk = row['spikes_aligned'][unit_idx]
            ax.eventplot([spk], lineoffsets=trial, linelengths=0.8, colors=[color])

            # Shade shared region for this trial
            t_al = row['t_aligned']
            sm = row['shared_mask']
            if sm.any():
                sh_start = t_al[sm][0]
                sh_end = t_al[sm][-1]
                ax.axvspan(sh_start, sh_end, ymin=(trial - 0.4) / (n_a + n_b),
                           ymax=(trial + 0.4) / (n_a + n_b),
                           color='green', alpha=0.08)
            trial += 1

    # Divider between clusters
    ax.axhline(n_a - 0.5, color='k', linewidth=0.5, linestyle='--')
    ax.set_ylabel('Trial')
    ax.set_title(f'Unit {unit_idx} (pType: {pType[unit_idx]}) — Clusters {ca} vs {cb}')

    # Shade shared region across full plot using median boundaries
    median_sh_start = df_aligned['t_aligned'].apply(
        lambda ta: ta[0]).median()  # rough; use 0 since aligned
    # Better: shade from t=0 to median shared duration
    median_sh_dur = df_aligned['shared_dur'].median()
    for ax_i in axes:
        ax_i.axvspan(0, median_sh_dur, color='green', alpha=0.08)
        ax_i.axvline(0, color='green', linewidth=0.8, linestyle='--', alpha=0.5)
        ax_i.axvline(median_sh_dur, color='red', linewidth=0.8, linestyle='--', alpha=0.5)

    # --- Position traces (x, y, z) ---
    dim_labels = ['X (m)', 'Y (m)', 'Z (m)']
    for dim in range(3):
        ax = axes[dim + 1]
        for cluster_flights, color in [(flights_a, pal[0]), (flights_b, pal[1])]:
            for _, row in cluster_flights.iterrows():
                ax.plot(row['t_aligned'], row['r'][:, dim],
                        color=color, alpha=0.3, linewidth=0.8)
            # Mean across flights
            # Interpolate to common time grid for mean
            t_common = np.linspace(all_t_min, all_t_max, 300)
            interped = []
            for _, row in cluster_flights.iterrows():
                f = interp1d(row['t_aligned'], row['r'][:, dim],
                             bounds_error=False, fill_value=np.nan)
                interped.append(f(t_common))
            mean_pos = np.nanmean(interped, axis=0)
            sem_pos = np.nanstd(interped, axis=0) / np.sqrt(len(interped))
            ax.plot(t_common, mean_pos, color=color, linewidth=2)
            ax.fill_between(t_common, mean_pos - sem_pos, mean_pos + sem_pos,
                            color=color, alpha=0.15)
        ax.set_ylabel(dim_labels[dim])

    axes[-1].set_xlabel('Time aligned to shared segment start (s)')
    plt.tight_layout()
    plt.show()


# Plot rasters for top units
for u in top_units:
    plot_raster_with_positions(df_aligned, u, ca, cb)

## Verify Position Alignment (Permutation Tests)\n\nTest that x, y, z positions do NOT significantly differ in the shared segment between the two clusters. This is a necessary check: if positions differ, any neural differences could be explained by place coding alone.

In [ ]:
BIN_DUR = 0.050  # 50 ms bins for PSTH and position comparison
N_PERMS = 1000
rng = np.random.default_rng(42)

# Bin the shared-segment positions into 50ms bins for each flight
# Use aligned time: shared segment is approximately [0, shared_dur]
median_sh_dur = df_aligned['shared_dur'].median()
bin_edges_sh = np.arange(0, median_sh_dur + BIN_DUR, BIN_DUR)
n_bins = len(bin_edges_sh) - 1
bin_centers = (bin_edges_sh[:-1] + bin_edges_sh[1:]) / 2

# Collect binned positions per cluster
def bin_positions_in_shared(flight_row, bin_edges):
    """Bin x,y,z positions within the shared segment into time bins."""
    t_al = flight_row['t_aligned']
    pos = flight_row['r']
    sm = flight_row['shared_mask']
    t_sh = t_al[sm]
    r_sh = pos[sm]
    n_bins = len(bin_edges) - 1
    binned = np.full((n_bins, 3), np.nan)
    for b in range(n_bins):
        mask = (t_sh >= bin_edges[b]) & (t_sh < bin_edges[b + 1])
        if mask.any():
            binned[b] = r_sh[mask].mean(axis=0)
    return binned  # (n_bins, 3)

pos_a = np.stack([bin_positions_in_shared(row, bin_edges_sh)
                  for _, row in df_aligned[df_aligned['cluster_id'] == ca].iterrows()])
pos_b = np.stack([bin_positions_in_shared(row, bin_edges_sh)
                  for _, row in df_aligned[df_aligned['cluster_id'] == cb].iterrows()])
# pos_a shape: (n_flights_a, n_bins, 3), pos_b: (n_flights_b, n_bins, 3)

# Observed mean difference (absolute) per bin per dimension
obs_diff = np.abs(np.nanmean(pos_a, axis=0) - np.nanmean(pos_b, axis=0))  # (n_bins, 3)

# Permutation test: shuffle cluster labels
all_pos = np.concatenate([pos_a, pos_b], axis=0)  # (n_total, n_bins, 3)
n_a_flights = len(pos_a)
n_total = len(all_pos)

perm_diffs = np.zeros((N_PERMS, n_bins, 3))
for p in range(N_PERMS):
    perm_idx = rng.permutation(n_total)
    perm_a = all_pos[perm_idx[:n_a_flights]]
    perm_b = all_pos[perm_idx[n_a_flights:]]
    perm_diffs[p] = np.abs(np.nanmean(perm_a, axis=0) - np.nanmean(perm_b, axis=0))

# P-value: fraction of permutations with difference >= observed
p_values_pos = np.mean(perm_diffs >= obs_diff[None, :, :], axis=0)  # (n_bins, 3)

# Summary: average p-value across bins per dimension
dim_names = ['X', 'Y', 'Z']
print("Position alignment test (shared segment):")
print("  Expect p >> 0.05 (no significant difference)")
for d, name in enumerate(dim_names):
    mean_p = np.nanmean(p_values_pos[:, d])
    min_p = np.nanmin(p_values_pos[:, d])
    print(f"  {name}: mean p={mean_p:.3f}, min p={min_p:.3f}")

# Also test velocity alignment
def bin_velocity_in_shared(flight_row, bin_edges):
    t_al = flight_row['t_aligned']
    vel = flight_row['v']
    sm = flight_row['shared_mask']
    t_sh = t_al[sm]
    v_sh = vel[sm]
    n_bins = len(bin_edges) - 1
    binned = np.full((n_bins, 3), np.nan)
    for b in range(n_bins):
        mask = (t_sh >= bin_edges[b]) & (t_sh < bin_edges[b + 1])
        if mask.any():
            binned[b] = v_sh[mask].mean(axis=0)
    return binned

vel_a = np.stack([bin_velocity_in_shared(row, bin_edges_sh)
                  for _, row in df_aligned[df_aligned['cluster_id'] == ca].iterrows()])
vel_b = np.stack([bin_velocity_in_shared(row, bin_edges_sh)
                  for _, row in df_aligned[df_aligned['cluster_id'] == cb].iterrows()])

obs_vel_diff = np.abs(np.nanmean(vel_a, axis=0) - np.nanmean(vel_b, axis=0))
all_vel = np.concatenate([vel_a, vel_b], axis=0)
perm_vel_diffs = np.zeros((N_PERMS, n_bins, 3))
for p in range(N_PERMS):
    perm_idx = rng.permutation(n_total)
    perm_vel_diffs[p] = np.abs(
        np.nanmean(all_vel[perm_idx[:n_a_flights]], axis=0) -
        np.nanmean(all_vel[perm_idx[n_a_flights:]], axis=0))

p_values_vel = np.mean(perm_vel_diffs >= obs_vel_diff[None, :, :], axis=0)
print("\nVelocity alignment test (shared segment):")
for d, name in enumerate(dim_names):
    mean_p = np.nanmean(p_values_vel[:, d])
    min_p = np.nanmin(p_values_vel[:, d])
    print(f"  {name}: mean p={mean_p:.3f}, min p={min_p:.3f}")

## Test Neural Coding Differences (PSTH Permutation Tests)\n\nFor each neuron, test whether the PSTH in the shared segment differs between the two clusters. If neurons are pure place coders, PSTHs should not differ (same position = same firing). Significant differences indicate non-local coding.

In [ ]:
from statsmodels.stats.multitest import multipletests

def compute_psth(flight_rows, unit_idx, bin_edges):
    """Compute PSTH (spike count per bin) for one unit across flights."""
    n_bins = len(bin_edges) - 1
    counts = np.zeros((len(flight_rows), n_bins))
    for i, (_, row) in enumerate(flight_rows.iterrows()):
        spk = row['spikes_aligned'][unit_idx]
        # Only count spikes in the shared segment [0, shared_dur]
        counts[i], _ = np.histogram(spk, bins=bin_edges)
    return counts  # (n_flights, n_bins)

# Compute PSTHs for all units in shared segment
flights_a = df_aligned[df_aligned['cluster_id'] == ca]
flights_b = df_aligned[df_aligned['cluster_id'] == cb]
n_a_fl, n_b_fl = len(flights_a), len(flights_b)

# Test statistic: sum of absolute differences in mean PSTH across bins
obs_stats = np.zeros(n_units)
psth_a_all = np.zeros((n_units, n_a_fl, n_bins))
psth_b_all = np.zeros((n_units, n_b_fl, n_bins))

for u in range(n_units):
    psth_a_all[u] = compute_psth(flights_a, u, bin_edges_sh)
    psth_b_all[u] = compute_psth(flights_b, u, bin_edges_sh)
    obs_stats[u] = np.sum(np.abs(psth_a_all[u].mean(axis=0) - psth_b_all[u].mean(axis=0)))

# Permutation test: shuffle cluster labels across flights
perm_stats = np.zeros((N_PERMS, n_units))
n_total_fl = n_a_fl + n_b_fl

for p in range(N_PERMS):
    perm_idx = rng.permutation(n_total_fl)
    for u in range(n_units):
        all_psth = np.concatenate([psth_a_all[u], psth_b_all[u]], axis=0)
        perm_a = all_psth[perm_idx[:n_a_fl]]
        perm_b = all_psth[perm_idx[n_a_fl:]]
        perm_stats[p, u] = np.sum(np.abs(perm_a.mean(axis=0) - perm_b.mean(axis=0)))

# P-values
p_values_neural = np.mean(perm_stats >= obs_stats[None, :], axis=0)

# FDR correction
reject, p_corrected, _, _ = multipletests(p_values_neural, method='fdr_bh', alpha=0.05)

n_sig = reject.sum()
print(f"Neural PSTH test (shared segment, {n_units} units):")
print(f"  {n_sig} / {n_units} neurons significant at FDR q<0.05")
print(f"  {(p_values_neural < 0.05).sum()} significant uncorrected at p<0.05")

# Also test divergent segment as positive control
bin_edges_div_before = np.arange(
    df_aligned['t_aligned'].apply(lambda ta: ta[0]).min(),
    0, BIN_DUR)
if len(bin_edges_div_before) > 2:
    obs_stats_div = np.zeros(n_units)
    for u in range(n_units):
        psth_a_div = compute_psth(flights_a, u, bin_edges_div_before)
        psth_b_div = compute_psth(flights_b, u, bin_edges_div_before)
        obs_stats_div[u] = np.sum(np.abs(
            psth_a_div.mean(axis=0) - psth_b_div.mean(axis=0)))

    perm_stats_div = np.zeros((N_PERMS, n_units))
    for p in range(N_PERMS):
        perm_idx = rng.permutation(n_total_fl)
        for u in range(n_units):
            psth_a_div = compute_psth(flights_a, u, bin_edges_div_before)
            psth_b_div = compute_psth(flights_b, u, bin_edges_div_before)
            all_psth_div = np.concatenate([psth_a_div, psth_b_div], axis=0)
            perm_stats_div[p, u] = np.sum(np.abs(
                all_psth_div[perm_idx[:n_a_fl]].mean(axis=0) -
                all_psth_div[perm_idx[n_a_fl:]].mean(axis=0)))

    p_values_div = np.mean(perm_stats_div >= obs_stats_div[None, :], axis=0)
    reject_div, _, _, _ = multipletests(p_values_div, method='fdr_bh', alpha=0.05)
    print(f"\nDivergent segment (positive control, pre-shared):")
    print(f"  {reject_div.sum()} / {n_units} neurons significant at FDR q<0.05")
    print(f"  {(p_values_div < 0.05).sum()} significant uncorrected at p<0.05")

In [ ]:
# Volcano-style plot: effect size vs -log10(p-value)
effect_sizes = obs_stats / (n_bins * BIN_DUR)  # normalize by duration

fig, ax = plt.subplots(figsize=(8, 5))
colors = np.where(reject, 'red', 'gray')
ax.scatter(effect_sizes, -np.log10(p_values_neural + 1e-10), c=colors, alpha=0.6, s=20)
ax.axhline(-np.log10(0.05), color='blue', linestyle='--', alpha=0.5, label='p=0.05 (uncorrected)')
ax.set_xlabel('Effect size (sum |PSTH diff| / duration)')
ax.set_ylabel('-log10(p-value)')
ax.set_title(f'Neural coding differences in shared segment\n'
             f'{n_sig}/{n_units} significant (FDR q<0.05, red)')
ax.legend()
sns.despine()
plt.tight_layout()
plt.show()

# Ranked bar plot of significant neurons
if n_sig > 0:
    sig_idx = np.where(reject)[0]
    sig_effects = effect_sizes[sig_idx]
    sort_order = np.argsort(sig_effects)[::-1]

    fig, ax = plt.subplots(figsize=(max(6, n_sig * 0.4), 4))
    colors_bar = [pal[2] if pType[sig_idx[i]] == 'pPYR' else pal[3]
                  for i in sort_order]
    ax.bar(range(len(sig_idx)), sig_effects[sort_order], color=colors_bar, alpha=0.8)
    ax.set_xticks(range(len(sig_idx)))
    ax.set_xticklabels([f'{sig_idx[i]}\n({pType[sig_idx[i]]})' for i in sort_order],
                       fontsize=7, rotation=45)
    ax.set_xlabel('Unit (cell type)')
    ax.set_ylabel('Effect size')
    ax.set_title('Significant non-local coders (shared segment)')
    sns.despine()
    plt.tight_layout()
    plt.show()

## Save Results

In [ ]:
import pickle, os

save_dir = os.path.join(os.path.dirname(os.getcwd()), 'tmp')
os.makedirs(save_dir, exist_ok=True)

results = dict(
    session_name=session_name,
    df_qualifying=df_qualifying,
    df_aligned=df_aligned,
    p_values_neural=p_values_neural,
    p_corrected=p_corrected,
    reject=reject,
    effect_sizes=effect_sizes,
    obs_stats=obs_stats,
    pType=pType,
    DIST_THRESHOLD=DIST_THRESHOLD,
    BIN_DUR=BIN_DUR,
    N_PERMS=N_PERMS,
)

save_path = os.path.join(save_dir, f'nonlocal_coding_{session_name}.pkl')
with open(save_path, 'wb') as f:
    pickle.dump(results, f)
print(f"Saved results to {save_path}")